In [ ]:
import atlasopenmagic as atom
import awkward as ak
import uproot
import numpy as np
import matplotlib.pyplot as plt
import vector

from itertools import combinations
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

# **Data Analysis II**
## *Lab 5: Multiclassification in HEP analyses*

Last time, we used a BDT classifier to discriminate signal processes from background ones. While this is one of the uses of Machine Learning in HEP, it's far from the only one. For example, we might want to see what process does an event look like? Does it look like a $H→4ℓ$? Like a $ZZ^*$ background? A $t\bar{t}$ pair production? We can do this by training a multiclassifier to distinguish between our three chosen samples.

We will use the same event-level kinematic features as in the previous lab, encode each process with its own label, train our model to predict all three classes and evaluate it with per-class metrics. 

The classifier is trained on MC samples only! Real data has no known process label so we can't train on it.



### *What does multiclassification mean?*

A binary classifier such as the one we built only thinks "signal or background?". Here, the model will return a probability for each of three classes. The predicted class is the one with the largest probability, with the three probabilities for an event having a sum of one.

We can label our samples like this:

| Label | Process class |
|---:|---|
| 0 | $Z$, $t\bar{t}$, $t\bar{t}+V$, $VVV$ background |
| 1 | $ZZ^*$ background |
| 2 | Higgs signal |

We use the same skim and MC samples definitions as last time.

In [ ]:
atom.set_release('2025e-13tev-beta')

defs = {
    r'Data':{'dids':['data']},
    r'Background $Z,t\bar{t},t\bar{t}+V,VVV$':{'dids': [364245, 364247, 410408,
                                                        410470, 410155, 410218,
                                                        410219, 410220, 412043, 
                                                        364242, 364243, 364246,
                                                        364248, 700320, 700321, 
                                                        700322, 700323, 700324, 
                                                        700325, 700587, 700792, 
                                                        700793, 700794]},
    r'Background $ZZ^{*}$':     {'dids': [700600, 700601]},
    r'Signal ($m_H$ = 125 GeV)':  {'dids': [345060, 346228, 346340, 
                                            346341, 346342, 345066, 
                                            346414, 346511]},
}

# We will use the 4 leptons in the final state skim
data_samples = atom.build_dataset(defs, skim='exactly4lep', protocol='root')

### *Pre-selection and MC weighting*

We will have to compute the MC weights and make the same basic cuts as we did last time.

In [ ]:
def calc_weight(data):
    weight_list = ( data["ScaleFactor_ELE"] * data["ScaleFactor_MUON"] * data["ScaleFactor_LepTRIGGER"] * 
                  data["ScaleFactor_PILEUP"] * ( data["mcWeight"] / data["sum_of_weights"]) * 
                  (data["xsec"] * data["filteff"] * data["kfac"] * lumi) )
    return weight_list

In [ ]:
# Check to see if the total charge of the leptons is zero
def cut_lep_charge(lep_charge):
    return ak.sum(lep_charge, axis=1) == 0

In [ ]:
# Check if either of the electron or the muon triggers fired
def cut_trig(trigE, trigM):
    return trigE | trigM

In [ ]:
# Check if at least one lepton in the event is matched with a trigger
def cut_isTrigMatched(lep_isTrigMatched):
    return ak.sum(lep_isTrigMatched, axis=1) >= 1

In [ ]:
# Cut on the correct identification of the particle and on whether the detected leptons are well isolated
def cut_ID_Iso(el_ID, mu_ID, el_Iso, mu_Iso, pid):
    return (ak.sum(((pid == 13) & mu_ID & mu_Iso) | ((pid == 11) & el_ID & el_Iso), axis=1) == 4)

In [ ]:
# Cut on the lepton type:
# 44 = e+e-e+e-
# 48 = e+e-mu+mu-
# 52 = mu+mu-mu+mu-
def cut_lep_type(lep_type):
    sum_lep_type = ak.sum(lep_type, axis=1)
    return (sum_lep_type == 44) | (sum_lep_type == 48) | (sum_lep_type == 52)

We will also want to make a function which computes the invariant mass of the 4 leptons.

In [ ]:
def calc_mass(lep_pt, lep_eta, lep_phi, lep_e):
    p4 = vector.zip({'pt': lep_pt, 
                     'eta': lep_eta, 
                     'phi': lep_phi, 
                     'E': lep_e})
    return ak.sum(p4, axis=1).M

We define the function that processes our file, makes the cuts and computes the invariant mass and the weights.

In [ ]:
def process_file(filename, sample_name):
    tree = uproot.open(filename + ':analysis')
    sample_data = []

    for data in tree.iterate(filter_name=variables, library='ak', entry_stop=int(fraction*tree.num_entries), step_size=100_000):
        
        data = data[cut_lep_charge(data["lep_charge"])]
        data = data[cut_trig(data["trigE"], data["trigM"])]
        data = data[cut_isTrigMatched(data["lep_isTrigMatched"])]
        data = data[cut_ID_Iso(data["lep_isLooseID"], data["lep_isMediumID"], data["lep_isLooseIso"], data["lep_isLooseIso"], data["lep_type"])]

        data['inv_mass'] = calc_mass(data["lep_pt"], data["lep_eta"], data["lep_phi"], data["lep_e"])

        if 'Data' not in sample_name:
            data["Weight"] = calc_weight(data)
        else:
            data["Weight"] = ak.ones_like(data["met"])  # Assign weight of 1 for data samples

        sample_data.append(data)

    if sample_data is not None and len(sample_data) > 0:
        return ak.concatenate(sample_data, axis=0) 
    else:
        return ak.Array([])

We will choose the variables, define the luminosity and start processing the data files. We define the `fraction` factor to be able to only go through a certain section of the datasets, as to not wait that long.

In [ ]:
variables = [
    "mcWeight", "ScaleFactor_LepTRIGGER", "trigE", "trigM", "lep_n", "lep_pt", 
    "lep_eta", "lep_phi", "lep_e", "lep_charge", "lep_type", "met", "met_phi", "lep_isMediumID", "lep_isLooseIso", 
    "sum_of_weights", "ScaleFactor_FTAG", "xsec", "lep_isTrigMatched", "eventNumber", "ScaleFactor_ELE", "ScaleFactor_MUON", 
    "ScaleFactor_PILEUP", "filteff", "kfac", "lep_isLooseID"
]

# Dictionary storing the data per process
data_all = {}
# Luminosity in pb^-1, you can find the luminosity of each dataset on the ATLAS Open Data Platform
lumi = 36600
# Fraction of the dataset to process
fraction = 1.0

for sample in data_samples:
    frames = []
    print(f"Processing the {sample} sample")

    for file in data_samples[sample]['list']:
        df = process_file(file, sample)
        if df is not None and len(df) > 0:
            frames.append(df)

    data_all[sample] = ak.concatenate(frames, axis=0)
    print(f'  Selected events: {len(data_all[sample])}')

Let's now take a look at the 4 lepton invariant masses. We will also label our data as we mentioned earlier, encoding each process type with a number in $\{0, 1, 2\}$.

In [ ]:
# We label our classes, store the data and give each of them a colour
bdt_classes = [
    {'name': r'Background $Z,t\bar{t},t\bar{t}+V,VVV$', 'data': data_all["Background $Z,t\\bar{t},t\\bar{t}+V,VVV$"], 'color': 'purple', 'label': 0},
    {'name': r'Background $ZZ^{*}$', 'data': data_all["Background $ZZ^{*}$"], 'color': 'red', 'label': 1},
    {'name': r'Signal ($m_H$ = 125 GeV)', 'data': data_all["Signal ($m_H$ = 125 GeV)"], 'color': 'cyan', 'label': 2},
]

bins_mass = np.arange(80, 250, 2.5)
bins_mass_centers = (bins_mass[1:] + bins_mass[:-1]) / 2
fig, ax = plt.subplots()
# We plot the MC samples
ax.hist([class_id['data']['inv_mass'] for class_id in bdt_classes], bins=bins_mass, weights=[class_id['data']['Weight'] for class_id in bdt_classes], stacked=True, label=[class_id['name'] for class_id in bdt_classes], color=[class_id['color'] for class_id in bdt_classes])

# We plot the real data points
data_x, _ = np.histogram(data_all['Data']['inv_mass'], bins=bins_mass)
data_x_err = np.sqrt(data_x)
ax.errorbar(bins_mass_centers, data_x, yerr=data_x_err, fmt='k.', label='Data',)

ax.set_xlabel(r'$m_{4l}$ [GeV]')
ax.set_ylabel(f'Events / {(bins_mass[1] - bins_mass[0]):.1f} GeV')
ax.legend()
fig.tight_layout()
plt.show()

### *Construct the BDT features*

The features we will use include:
 - The four lepton $pT$'s, ordered from highest to lowest
 - Their $η$
 - The $ΔR$ between each pair of leptons
 - The $MET$

In [ ]:
def make_bdt_features(sample):
    # We want to sort the observables from the highest pT to the fourth highest over all variables for easy work
    order = ak.argsort(sample['lep_pt'], axis=1, ascending=False)
    pt = ak.to_numpy(ak.to_regular(sample['lep_pt'][order], axis=1))
    eta = ak.to_numpy(ak.to_regular(sample['lep_eta'][order], axis=1))
    phi = ak.to_numpy(ak.to_regular(sample['lep_phi'][order], axis=1))

    # We compute the difference in azimuthal angle
    delta_phi = np.abs(np.arctan2(
        np.sin(phi[:, :, None] - phi[:, None, :]),
        np.cos(phi[:, :, None] - phi[:, None, :]),
    ))
    # We compute the difference in pseudorapidity
    delta_eta = eta[:, :, None] - eta[:, None, :]
    # We do pairwise combinations of angular separation
    delta_r = np.sqrt(delta_eta**2 + delta_phi**2)
    pairs = list(combinations(range(4), 2))
    pair_delta_r = np.column_stack([delta_r[:, i, j] for i, j in pairs])

    # We take the MET as a feature as well
    met = np.asarray(ak.to_numpy(sample['met']))

    # We concatenate the features together and pass them back
    features = np.column_stack((pt, eta, pair_delta_r, met))
    return features

We will take each class, build the BDT features, make an array holding the labels and hold the weights for the events as well.

In [ ]:
class_ids = np.array([class_id['label'] for class_id in bdt_classes])
features = []
labels = []
weights = []
# We parse the classes, build the feature matrix, the label vector and the weight vector
for class_id in bdt_classes:
    features_sample = make_bdt_features(class_id['data'])
    features.append(features_sample)
    labels.append(np.full(len(features_sample), class_id['label'], dtype=int))
    weights.append(ak.to_numpy(class_id['data']['Weight']))

# We go from per-class nested lists to larger flat arrays
X = np.concatenate(features, axis=0)
y = np.concatenate(labels)
all_weights = np.concatenate(weights)

# We can take a look at the number of events and features
print('Feature matrix shape:', X.shape)
# And also look at how many events per class we have
print('Events per class:', {class_id['name']: int(np.count_nonzero(y == class_id['label'])) for class_id in bdt_classes})

Now we will have to do the train-validation-test split. We will split the data as follows:
 - Training: 60%
 - Validation: 20% 
 - Testing: 20%

After that, we compute the training weights per event.

In [ ]:
# We split total into train+validation and test
(X_train_val, X_test,
 y_train_val, y_test,
 weights_train_val, weights_test) = train_test_split(X, y, all_weights,
                                                     test_size=0.2,
                                                     stratify=y,)

# We split train+validation into train and validation
(X_train, X_val,
 y_train, y_val,
 weights_train, weights_val) = train_test_split(X_train_val, y_train_val, weights_train_val,
                                                test_size=0.25,
                                                stratify=y_train_val)

# We compute the number of events for each class
train_class_counts = np.bincount(y_train, minlength=len(class_ids))

# We compute per event weights, which should be the same per class and sum to 1.0
train_weights = np.zeros(len(y_train), dtype=float)
for class_id in class_ids:
    train_weights[y_train == class_id] = 1.0 / (len(class_ids) * train_class_counts[class_id])
# We can check to see how the weights look like 
print('Total training weight per class:', {class_id['name']: float(train_weights[y_train == class_id['label']].sum()) for class_id in bdt_classes})

We can now train our BDT classifier!

In [ ]:
bdt = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    random_state=42,
)
bdt.fit(X_train, y_train, sample_weight=train_weights)

We can now use the validation sample to compare hyperparameters or feature choices. The report contains unweighted event-level metrics. Once we are satisfied with the model, we fix it and use the test sample below for the final evaluation.

In [ ]:
# Score the validation set with the trained model
val_probabilities = bdt.predict_proba(X_val)
val_predictions = bdt.classes_[np.argmax(val_probabilities, axis=1)]
# We will look at some performance metrics for the validation set
print(classification_report(
    y_val,
    val_predictions,
    labels=class_ids,
    target_names=[class_id['name'] for class_id in bdt_classes],
    zero_division=0,
))

We will build three confusion matrices which show raw test event counts, event fractions normalized to unity (both of which describe classifier behaviour) and MC-weighted expected yields (which accounts for the simulated process rates).

In [ ]:
# We will use the test sample for the final model evaluation. We predict the process probabilities and choose the most likely class for each event
test_probabilities = bdt.predict_proba(X_test)
test_predictions = bdt.classes_[np.argmax(test_probabilities, axis=1)]

# Summarize test performance with precision, recall and F1 score for each class
print(classification_report(
    y_test,
    test_predictions,
    labels=class_ids,
    target_names=[class_id['name'] for class_id in bdt_classes],
    zero_division=0,
))

# Scale the MC weights to estimate the full selected sample yields
test_class_fractions = {
    class_id['label']: np.count_nonzero(y_test == class_id['label']) / np.count_nonzero(y == class_id['label'])
    for class_id in bdt_classes
}
scaled_test_weights = weights_test / np.array([
    test_class_fractions[class_id] for class_id in y_test
])

# Compare raw event counts, fractions within each true class, and expected MC yields
cm_counts = confusion_matrix(y_test, test_predictions, labels=class_ids)
cm_fraction = confusion_matrix(y_test, test_predictions, labels=class_ids, normalize='true')
cm_yields = confusion_matrix(
    y_test, test_predictions, labels=class_ids, sample_weight=scaled_test_weights,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, matrix, title, fmt in [
    (axes[0], cm_counts, 'Test event counts', 'd'),
    (axes[1], cm_fraction, 'Fraction of each true class', '.2f'),
    (axes[2], cm_yields, 'MC-weighted expected yields', '.1f'),
]:
    # Add the values for each matrix
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_xticks(class_ids, [class_id['name'] for class_id in bdt_classes], rotation=30, ha='right')
    ax.set_yticks(class_ids, [class_id['name'] for class_id in bdt_classes])
    for i in range(len(class_ids)):
        for j in range(len(class_ids)):
            ax.text(j, i, format(matrix[i, j], fmt), ha='center', va='center')
    fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()

A multiclass model has no single binary ROC curve. Instead, we can evaluate each class in a one-vs-rest way: for a given class, compare its probability with the combined probability of the other classes.

In [ ]:
# Compute one-vs-rest AUCs
class_aucs = roc_auc_score(
    y_test,
    test_probabilities,
    labels=class_ids,
    multi_class='ovr',
    average=None,
)
print('One-vs-rest test AUC by class:')
for class_id, auc in zip(bdt_classes, class_aucs):
    print(f'  {class_id["name"]}: {auc:.3f}')

# Plot the ROC curves for each class
fig, ax = plt.subplots()
for class_id, auc in zip(bdt_classes, class_aucs):
    fpr, tpr, _ = roc_curve(y_test == class_id['label'], test_probabilities[:, class_id['label']])
    ax.plot(fpr, tpr, label=f'{class_id["name"]} (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_xlabel('False positive rate (one-vs-rest)')
ax.set_ylabel('True positive rate')
ax.legend()
fig.tight_layout()
plt.show()

We can also take a look at the feature importance of each of the 15 features that we use to see the impact each of them had in training.

In [ ]:
# Rank the input features by their contribution to the trained BDT
importance_order = np.argsort(bdt.feature_importances_)[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
feature_names = (
    ['pT1', 'pT2', 'pT3', 'pT4', 'eta1', 'eta2', 'eta3', 'eta4']
    + ['dR12', 'dR13', 'dR14', 'dR23', 'dR24', 'dR34']
    + ['MET']
)
# Plot the ten most important features
ax.barh(
    [feature_names[index] for index in importance_order[:10]][::-1],
    bdt.feature_importances_[importance_order[:10]][::-1],
)
ax.set_xlabel('Feature importance')
ax.set_title('Top BDT features')
fig.tight_layout()
plt.show()

### *Apply the classifier to data*

In this final step we look at how the trained model can be used on events for which the process is unknown. The largest-probability class is only the model's best match to the MC training classes, it is not a truth label for the real data. 

In [ ]:
# Apply the trained classifier to data, which has unknown labels
data_features = make_bdt_features(data_all['Data'])
data_probabilities = bdt.predict_proba(data_features)
data_predictions = bdt.classes_[np.argmax(data_probabilities, axis=1)]

# Count how many data events are assigned to each most-likely process class
print('Data events assigned to each most-likely class:')
for class_id in bdt_classes:
    print(f'  {class_id["name"]}: {np.count_nonzero(data_predictions == class_id["label"])}')

# Plot each class probability across all data events
fig, ax = plt.subplots()
score_bins = np.linspace(0, 1, 41)
for class_id in bdt_classes:
    ax.hist(
        data_probabilities[:, class_id['label']],
        bins=score_bins,
        histtype='step',
        label=f'P({class_id["name"]})',
    )
ax.set_xlabel('Predicted class probability for data events')
ax.set_ylabel('Events')
ax.legend()
fig.tight_layout()
plt.show()

### *What to do now?*

Here are some things to consider:
 - Which pair of classes is most often confused? Does that agree with the physics similarity of the processes?
 - Compare event count and MC-weighted confusion matrices. Why do they differ?
 - Change the BDT hyperparameters or the training class weighting and compare the metrics.
 - The model does not use $m_{4l}$ as a feature, but some of its input variables are correlated to it. Compare the mass spectrum before and after the event selection. Does the selection sculpt the mass spectrum?
 - Try different features! Which observables matter most, are the feature importances stable?